# Agente GAIA – Unidad 4 del Hugging Face Agents Course

Agente basado en **smolagents** + **Claude (Anthropic)** que responde a las 20 preguntas de nivel 1 del benchmark GAIA y envía las respuestas a la API de puntuación del curso.

**Cómo usarlo:**
1. En Colab, abre el panel 🔑 *Secrets* (barra izquierda) y crea un secreto llamado `ANTHROPIC_API_KEY` con tu clave. Activa "Notebook access".
2. Rellena la celda **Configuración** (tu usuario de Hugging Face y el enlace a tu repositorio de GitHub).
3. Ejecuta las celdas en orden. Primero prueba con 1 pregunta; luego lanza las 20.
4. Revisa las respuestas y ejecuta la celda **Enviar**.

## 1. Instalación

In [ ]:
!pip install -q "smolagents[litellm]" ddgs markdownify wikipedia-api youtube-transcript-api openpyxl transformers accelerate

## 2. Configuración

In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

# --- Rellena esto ---
HF_USERNAME = "jorgerollon-ia"                                   # tu usuario de Hugging Face
AGENT_CODE_URL = "https://github.com/Josulo72/gaia-agent"        # repositorio PÚBLICO con este notebook

# Modelo. Sonnet = mejor resultado; Haiku = más barato.
MODEL_ID = "anthropic/claude-sonnet-5-5"
# MODEL_ID = "anthropic/claude-haiku-5-5"

API_URL = "https://agents-course-unit4-scoring.hf.space"
FILES_DIR = "gaia_files"
os.makedirs(FILES_DIR, exist_ok=True)

## 3. Herramientas personalizadas

Además de búsqueda web, visitar páginas y Wikipedia, el agente puede transcribir audio, analizar imágenes, leer subtítulos de YouTube y ejecutar archivos `.py`.

In [ ]:
import base64, mimetypes, subprocess, re
import litellm
from smolagents import tool

_asr = None

@tool
def transcribe_audio(file_path: str) -> str:
    """Transcribe an audio file (mp3, wav, m4a...) to text using Whisper.

    Args:
        file_path: Local path to the audio file.
    """
    global _asr
    if _asr is None:
        from transformers import pipeline
        _asr = pipeline("automatic-speech-recognition", model="openai/whisper-small", chunk_length_s=30)
    return _asr(file_path, return_timestamps=True)["text"]


@tool
def analyze_image(file_path: str, question: str) -> str:
    """Look at an image file and answer a question about it (chess positions, charts, photos, text in images...).

    Args:
        file_path: Local path to the image (png, jpg...).
        question: What you want to know about the image. Be specific.
    """
    mime = mimetypes.guess_type(file_path)[0] or "image/png"
    with open(file_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    resp = litellm.completion(
        model=MODEL_ID,
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
            {"type": "text", "text": question},
        ]}],
        max_tokens=1500,
    )
    return resp.choices[0].message.content


@tool
def youtube_transcript(url: str) -> str:
    """Get the transcript (subtitles) of a YouTube video. May fail if the video has no subtitles.

    Args:
        url: Full YouTube URL.
    """
    m = re.search(r"(?:v=|youtu\.be/)([\w-]{11})", url)
    if not m:
        return "Could not find a video id in the URL."
    vid = m.group(1)
    try:
        from youtube_transcript_api import YouTubeTranscriptApi
        try:
            fetched = YouTubeTranscriptApi().fetch(vid)          # versión >= 1.0
            return " ".join(s.text for s in fetched)
        except AttributeError:
            data = YouTubeTranscriptApi.get_transcript(vid)      # versiones antiguas
            return " ".join(s["text"] for s in data)
    except Exception as e:
        return f"Transcript not available: {e}"


@tool
def run_python_file(file_path: str) -> str:
    """Execute a Python file and return everything it prints (stdout and stderr).

    Args:
        file_path: Local path to the .py file.
    """
    try:
        r = subprocess.run(["python", file_path], capture_output=True, text=True, timeout=60)
        return (r.stdout + "\n" + r.stderr).strip()
    except subprocess.TimeoutExpired:
        return "Execution timed out after 60 seconds."

## 4. El agente

In [ ]:
from smolagents import (CodeAgent, LiteLLMModel, DuckDuckGoSearchTool,
                        VisitWebpageTool, WikipediaSearchTool)

litellm.drop_params = True  # ignora parámetros que el modelo no admite
model = LiteLLMModel(model_id=MODEL_ID, max_tokens=4096)

agent = CodeAgent(
    tools=[
        DuckDuckGoSearchTool(),
        VisitWebpageTool(max_output_length=10000),  # limita el texto de cada página (ahorra tokens)
        WikipediaSearchTool(),
        transcribe_audio,
        analyze_image,
        youtube_transcript,
        run_python_file,
    ],
    model=model,
    additional_authorized_imports=["pandas", "numpy", "openpyxl", "json", "csv", "re",
                                   "math", "datetime", "collections", "itertools", "unicodedata"],
    max_steps=12,
    planning_interval=4,
    verbosity_level=1,
)

# Reglas de formato de GAIA: la corrección es por coincidencia EXACTA.
ANSWER_RULES = """
You are answering a question from the GAIA benchmark. Work step by step with your tools, verify facts, then call final_answer.
The value you pass to final_answer is compared by EXACT MATCH, so it must contain ONLY the answer:
- A number: digits only, no thousands separators, no units ($, %, km...) unless the question asks for them.
- A string: as few words as possible, no articles, no abbreviations, no trailing period, digits as numerals unless told otherwise.
- A list: comma-separated values following the rules above, in the order or sorting the question asks for.
- Never add explanations, "The answer is", quotes or extra words.
Follow any explicit formatting instruction in the question (alphabetical order, rounding, units, IOC codes...).
"""

def build_prompt(q, file_path=None):
    prompt = ANSWER_RULES + "\n\nQUESTION:\n" + q["question"]
    if file_path:
        prompt += (f"\n\nAn attached file is available at local path: {file_path}\n"
                   "Open it with the right method (pandas for xlsx/csv, transcribe_audio for audio, "
                   "analyze_image for images, run_python_file or open() for .py, open() for text).")
    return prompt

def clean(ans):
    """Deja solo la respuesta final, aunque el agente haya devuelto texto largo (p. ej. al agotar pasos)."""
    s = str(ans).strip()
    m = re.search(r"final_answer\(\s*[\"']?(.*?)[\"']?\s*\)", s, flags=re.S)
    if m:
        s = m.group(1)
    if len(s) > 80 or "\n" in s:
        # Respuesta larga: pedimos al modelo que extraiga solo el valor final
        resp = litellm.completion(model=MODEL_ID, max_tokens=100, messages=[{"role": "user", "content":
            "Extract ONLY the final answer value from this text, with no extra words, following GAIA formatting "
            "(numbers without units or commas, short strings, comma-separated lists). Text:\n\n" + s}])
        s = resp.choices[0].message.content
    s = re.sub(r"^(final answer|answer)\s*(is)?\s*[:\-]?\s*", "", s.strip(), flags=re.I)
    return s.strip().strip('"').strip("'").rstrip(".").strip()

## 5. Descargar preguntas y archivos

In [ ]:
import requests

questions = requests.get(f"{API_URL}/questions", timeout=30).json()
print(f"{len(questions)} preguntas")

def get_file(q):
    """Descarga el archivo adjunto de la pregunta. Si el servidor no lo tiene, devuelve None."""
    if not q.get("file_name"):
        return None
    path = os.path.join(FILES_DIR, q["file_name"])
    if not os.path.exists(path):
        try:
            r = requests.get(f"{API_URL}/files/{q['task_id']}", timeout=60)
            r.raise_for_status()
        except Exception as e:
            print(f"  ⚠️ No se pudo descargar {q['file_name']}: {e}")
            return None
        with open(path, "wb") as f:
            f.write(r.content)
    return path

for i, q in enumerate(questions):
    fp = get_file(q)
    print(f"{i:2d} | {q['question'][:90]!r} | archivo: {fp}")

## 6. Probar con una sola pregunta

Cambia el índice para probar distintas preguntas antes de gastar en las 20.

In [ ]:
q = questions[0]
fp = get_file(q)
print(q["question"], "\n")
answer = clean(agent.run(build_prompt(q, fp)))
print("\nRESPUESTA:", answer)

## 7. Ejecutar las preguntas en tandas

Cada ejecución resuelve 5 preguntas pendientes (controla el gasto). Las respuestas se guardan en `answers.json`; vuelve a ejecutar la celda para la siguiente tanda.

In [ ]:
import json, time

ANSWERS_FILE = "answers.json"
BATCH_SIZE = 5   # preguntas por tanda; vuelve a ejecutar la celda para la siguiente tanda

answers = json.load(open(ANSWERS_FILE)) if os.path.exists(ANSWERS_FILE) else {}

done_now = 0
for i, q in enumerate(questions):
    tid = q["task_id"]
    if tid in answers:
        continue
    if done_now >= BATCH_SIZE:
        break
    print(f"\n===== Pregunta {i+1}/{len(questions)} =====\n{q['question'][:200]}")
    try:
        ans = clean(agent.run(build_prompt(q, get_file(q))))
    except Exception as e:
        print("ERROR:", e)
        ans = ""
    answers[tid] = ans
    json.dump(answers, open(ANSWERS_FILE, "w"), ensure_ascii=False, indent=2)
    print("RESPUESTA:", ans)
    done_now += 1
    time.sleep(2)

print(f"\nRespondidas: {sum(1 for a in answers.values() if a)}/{len(questions)}")

## 8. Revisar respuestas

Si alguna sale vacía o con formato raro, puedes repetirla: borra su entrada de `answers` y vuelve a ejecutar la celda 7.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.DataFrame([{"#": i, "pregunta": q["question"][:120], "respuesta": answers.get(q["task_id"], "")}
              for i, q in enumerate(questions)])

In [ ]:
# Para repetir una pregunta concreta (cambia el índice) y luego vuelve a ejecutar la celda 7:
# del answers[questions[3]["task_id"]]; json.dump(answers, open(ANSWERS_FILE, "w"), ensure_ascii=False, indent=2)

## 9. Enviar a la API de puntuación

Necesitas un **30% o más** (6 de 20) para el certificado de excelencia.

In [ ]:
assert "TU_USUARIO" not in AGENT_CODE_URL, "Pon el enlace real a tu repositorio de GitHub en la celda de Configuración"

payload = {
    "username": HF_USERNAME,
    "agent_code": AGENT_CODE_URL,
    "answers": [{"task_id": tid, "submitted_answer": a} for tid, a in answers.items()],
}
r = requests.post(f"{API_URL}/submit", json=payload, timeout=120)
print(r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))